# Hands-on law tutorial

This notebook is a hands-on introduction to [law](https://github.com/riga/law) for newcomers.
It is organized in stages that build on each other, each introducing a few new concepts:

1. [A first task](#Stage-1:-A-first-task): outputs, `run()`, targets and formatters
2. [Parameters](#Stage-2:-Parameters): configurable tasks
3. [Dependencies](#Stage-3:-Dependencies): `requires()`, `input()`, `req()` and the task status
4. [A base task](#Stage-4:-A-base-task): sharing functionality and versioning outputs
5. [Workflows](#Stage-5:-Workflows): processing many similar tasks
6. [Inspecting and cleaning up](#Stage-6:-Inspecting-and-cleaning-up): interactive command line parameters
7. [Writing outputs robustly](#Stage-7:-Writing-outputs-robustly): failures, `localize()` and decorators
8. [Where to go from here](#Stage-8:-Where-to-go-from-here)

Throughout the tutorial, we simulate rolling dice and analyze the results.
The payload is deliberately simple, so that you can focus on how the tasks are structured.

Just execute the cells one after another.
Feel free to change parameters and code, and to execute cells multiple times, which is part of the learning experience.

## Setup

The next cell makes sure law is available.
When this notebook is opened inside a checkout of the law repository, law is taken from there, and otherwise it is installed.
It also fetches the `law.cfg` file of this tutorial if missing, and defines the directory in which all outputs are stored.

In [ ]:
import os
import sys
import urllib.request

# when running inside a checkout of the law repository, use law from there, and otherwise install it if necessary
checkout_dir = os.path.normpath(os.path.join(os.getcwd(), "..", ".."))
if os.path.isdir(os.path.join(checkout_dir, "src", "law")):
    sys.path.insert(0, os.path.join(checkout_dir, "src"))
else:
    try:
        import law
    except ImportError:
        %pip install -q law

# fetch the law config file of this tutorial if missing, e.g. on Colab
if not os.path.exists("law.cfg"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/riga/law/master/examples/tutorial/law.cfg", "law.cfg")

# all outputs of this tutorial are stored in this directory
os.environ["TUTORIAL_DATA_PATH"] = os.path.join(os.getcwd(), "data")

Now, we import [luigi](https://luigi.readthedocs.io), on which law is built, and law itself.

We also load the `ipython` *contrib* package of law, which registers two *magic* commands:

- `%ilaw` runs law commands, such as `%ilaw run SomeTask`, interactively within this notebook, so that it knows about the tasks we define here.
- `%law` runs law commands in a subprocess, just like on the command line.

We will only use `%ilaw`.

In [ ]:
import luigi

import law

law.contrib.load("ipython")

---

## Stage 1: A first task

A task is a unit of work that produces **outputs**.
In law, outputs are described by **targets**, which are objects that refer to files or directories, such as `law.LocalFileTarget`.

A task is **complete** when all its outputs exist.
This is the central idea of luigi and law: instead of tracking which code ran, tasks are considered done when their outputs are there, so that only missing outputs are produced.

A minimal task defines two methods:

- `output()` returns the target(s) of the task.
- `run()` contains the actual payload and creates the outputs.

In [ ]:
import random


class RollDice(law.ipython.Task):

    def output(self):
        return law.LocalFileTarget("$TUTORIAL_DATA_PATH/stage1/rolls.json")

    def run(self):
        rolls = [random.randint(1, 6) for _ in range(100)]
        self.output().dump(rolls)

A few notes:

- In notebooks, tasks inherit from `law.ipython.Task` so that you can redefine them by executing a cell again.
  In regular Python modules, you would use `law.Task` instead.
- Environment variables in target paths, such as `$TUTORIAL_DATA_PATH`, are expanded.
- `dump()` writes data into the target using a **formatter**.
  The json formatter is selected automatically based on the file extension, and `load()` works the same way.
  There are also formatters for text, yaml, pickle, numpy, pandas, and many more.

Let's run the task.

In [ ]:
%ilaw run RollDice

The output consists of three phases:

1. **Scheduling**: luigi builds the tree of tasks to run and checks which ones are complete, ending with `Done scheduling tasks`.
2. **Running**: incomplete tasks are executed.
3. **Summary**: an overview of what happened.

The `True` at the very end is the return value of `%ilaw`, which tells whether the run was successful.

Now, run the very same command again.

In [ ]:
%ilaw run RollDice

This time, nothing ran and the summary says the task was already complete, since its output exists.

Tasks and targets are regular Python objects, so you can also work with them directly, which is helpful for debugging and for looking at results:

In [ ]:
task = RollDice()
target = task.output()

print(target)
print(target.abspath)
print(target.exists())

rolls = target.load()
print(len(rolls), rolls[:10])

---

## Stage 2: Parameters

Our first task always does the same thing.
To make it configurable, tasks can have **parameters**, which are defined as class attributes.

Each combination of parameter values defines a separate task instance, and thus, **the outputs must depend on the parameter values**.
Otherwise, different instances would write to the same file, and the first one would make all others appear complete.

In [ ]:
class RollDice(law.ipython.Task):

    n_rolls = luigi.IntParameter(default=100, description="number of rolls; default: 100")
    n_sides = luigi.IntParameter(default=6, description="number of sides of the dice; default: 6")
    seed = luigi.IntParameter(default=1, description="random seed; default: 1")

    def output(self):
        return law.LocalFileTarget(
            f"$TUTORIAL_DATA_PATH/stage2/rolls_{self.n_sides}sides_{self.n_rolls}rolls_seed{self.seed}.json",
        )

    def run(self):
        # use a random generator with a fixed seed for reproducible results
        rnd = random.Random(self.seed)
        rolls = [rnd.randint(1, self.n_sides) for _ in range(self.n_rolls)]
        self.output().dump(rolls)

On the command line, parameters are passed with dashes instead of underscores.
Parameters that are not passed take their default values.

In [ ]:
%ilaw run RollDice --n-rolls 1000 --seed 2

Again, we can instantiate the task in Python with the same parameters to look at its output.
Note that luigi caches task instances, so creating a task with the same parameters twice returns the identical object.

In [ ]:
task = RollDice(n_rolls=1000, seed=2)
print(task)
print(task.output().load()[:10])

print(task is RollDice(n_rolls=1000, seed=2))
print(task is RollDice(n_rolls=1000, seed=3))

**Try it:** Run `RollDice` with a 20-sided dice (`--n-sides 20`) and look at its output.

---

## Stage 3: Dependencies

Tasks become really useful when they depend on each other.
A task declares the tasks it depends on in `requires()`, and accesses their outputs via `self.input()`.
When a task is run, luigi makes sure that all its requirements are complete first, and runs them otherwise.

Let's add two tasks:

- `CountFaces` requires `RollDice` and counts how often each face was rolled.
- `PlotHistogram` requires `CountFaces` and draws a text histogram.

Both need the same parameters as `RollDice`.
Instead of defining them again, they are simply reused, e.g. `n_rolls = RollDice.n_rolls`, which copies the parameter definition, not the value.

Requirements are created via `RollDice.req(self)`.
`req()` creates an instance of `RollDice` and passes all parameter values of `self` that `RollDice` knows about, so that you do not need to forward them one by one.

In [ ]:
class CountFaces(law.ipython.Task):

    n_rolls = RollDice.n_rolls
    n_sides = RollDice.n_sides
    seed = RollDice.seed

    def requires(self):
        return RollDice.req(self)

    def output(self):
        return law.LocalFileTarget(
            f"$TUTORIAL_DATA_PATH/stage3/counts_{self.n_sides}sides_{self.n_rolls}rolls_seed{self.seed}.json",
        )

    def run(self):
        # self.input() is the output of the required task
        rolls = self.input().load()

        counts = {face: rolls.count(face) for face in range(1, self.n_sides + 1)}
        self.output().dump(counts, indent=4)


class PlotHistogram(law.ipython.Task):

    n_rolls = RollDice.n_rolls
    n_sides = RollDice.n_sides
    seed = RollDice.seed

    def requires(self):
        return CountFaces.req(self)

    def output(self):
        return law.LocalFileTarget(
            f"$TUTORIAL_DATA_PATH/stage3/histogram_{self.n_sides}sides_{self.n_rolls}rolls_seed{self.seed}.txt",
        )

    def run(self):
        # note that json converts the integer keys of the counts into strings
        counts = self.input().load()
        max_count = max(counts.values())

        lines = [
            f"{face:>3} | {'#' * round(40 * count / max_count)} {count}"
            for face, count in counts.items()
        ]
        histogram = "\n".join(lines)
        self.output().dump(histogram + "\n", formatter="text")

        # publish_message prints the message, and also sends it to a central scheduler if one is used
        self.publish_message(histogram)

Before running anything, we can ask law for the **status** of a task and all its dependencies.
`--print-status -1` checks all outputs recursively, whereas a positive number limits the depth.
Note that `law run` only prints the status and does not run anything when this parameter is set.

In [ ]:
%ilaw run PlotHistogram --n-rolls 600 --print-status -1

Nothing exists yet.
Now let's run the `PlotHistogram` task, which triggers the other two tasks as well.

In [ ]:
%ilaw run PlotHistogram --n-rolls 600

Check the status again.
All outputs should exist now.

In [ ]:
%ilaw run PlotHistogram --n-rolls 600 --print-status -1

**Try it:** Run `PlotHistogram` with `--n-rolls 1000 --seed 2`.
Which tasks actually run, and why?
<details><summary>Answer</summary>
Only <code>CountFaces</code> and <code>PlotHistogram</code> run, since the output of <code>RollDice</code> with these parameters was already created in stage 2.
</details>

---

## Stage 4: A base task

You might have noticed some repetition above: each task defines the same parameters and builds similar output paths.
In real projects, tasks usually inherit from a common **base task** that provides shared functionality.

Below, `TutorialTask` adds two things:

- A `version` parameter, which becomes part of all output paths.
  When you change the implementation of a task, you can bump the version to produce new outputs while keeping the old ones, instead of deleting them.
- A `local_target()` method that places outputs in a directory per task and version, given by `store_parts()`.

On top of that, `DiceTask` defines the dice parameters and a helper that encodes their values into file names.

In [ ]:
class TutorialTask(law.ipython.Task):

    version = luigi.Parameter(default="v1", description="version of the outputs to produce; default: v1")

    def store_parts(self):
        return (self.task_family, self.version)

    def local_target(self, *path):
        parts = ("$TUTORIAL_DATA_PATH", "stage4", *self.store_parts(), *path)
        return law.LocalFileTarget(os.path.join(*map(str, parts)))


class DiceTask(TutorialTask):

    n_rolls = luigi.IntParameter(default=100, description="number of rolls; default: 100")
    n_sides = luigi.IntParameter(default=6, description="number of sides of the dice; default: 6")
    seed = luigi.IntParameter(default=1, description="random seed; default: 1")

    def dice_postfix(self):
        return f"{self.n_sides}sides_{self.n_rolls}rolls_seed{self.seed}"

With these base tasks, the actual tasks become much shorter and only contain what is specific to them.

In [ ]:
class RollDice(DiceTask):

    def output(self):
        return self.local_target(f"rolls_{self.dice_postfix()}.json")

    def run(self):
        rnd = random.Random(self.seed)
        self.output().dump([rnd.randint(1, self.n_sides) for _ in range(self.n_rolls)])


class CountFaces(DiceTask):

    def requires(self):
        return RollDice.req(self)

    def output(self):
        return self.local_target(f"counts_{self.dice_postfix()}.json")

    def run(self):
        rolls = self.input().load()
        self.output().dump({face: rolls.count(face) for face in range(1, self.n_sides + 1)}, indent=4)

In [ ]:
%ilaw run CountFaces --n-rolls 500

Let's have a look at the output paths, which now contain the task name and the version.
`--print-output` prints a flat list of the outputs of a task and its dependencies up to a certain depth.

In [ ]:
%ilaw run CountFaces --n-rolls 500 --print-output 1

**Try it:** Check the status of `CountFaces` with `--version v2`.
Since outputs of different versions are stored separately, nothing exists yet for the new version.

---

## Stage 5: Workflows

Often, you want to run the same task many times with slightly different inputs, e.g. on many files or, in our case, for many experiments with different seeds.
For that, law provides **workflows**.

A workflow is a task with a **branch map** that defines which data each **branch** processes.
The same class represents both the workflow and its branches, distinguished by the `branch` parameter:

- `branch == -1` (default): the task is *the workflow*, and its output is the collection of all branch outputs.
  Running it means running all its branches.
- `branch >= 0`: the task is *a branch*, which uses `self.branch_data` to know what to process.

`output()` and `run()` are written from the perspective of a single branch.
Below, each branch rolls dice with a different seed, given by its branch data.

In [ ]:
class RollDiceExperiments(DiceTask, law.LocalWorkflow):

    n_experiments = luigi.IntParameter(default=10, description="number of experiments; default: 10")

    # the seed is defined by the branch map, so remove the parameter
    seed = None

    def create_branch_map(self):
        # map branch numbers to seeds
        return {b: 100 + b for b in range(self.n_experiments)}

    def output(self):
        return self.local_target(f"rolls_{self.n_sides}sides_{self.n_rolls}rolls_{self.branch}.json")

    def run(self):
        rnd = random.Random(self.branch_data)
        self.output().dump([rnd.randint(1, self.n_sides) for _ in range(self.n_rolls)])

Single branches can be run with `--branch`.

In [ ]:
%ilaw run RollDiceExperiments --branch 0

When running the workflow, `--branches` can select a subset of branches, given as a comma-separated list of numbers and ranges.
Without it, all branches are run, but those that are complete are skipped, such as branch 0 below.

In [ ]:
%ilaw run RollDiceExperiments --branches 1:4 --print-status 0

In [ ]:
%ilaw run RollDiceExperiments

Other tasks usually require the workflow rather than single branches.
The output of a workflow is a dictionary whose `"collection"` entry holds a **target collection** with the outputs of all branches.

In [ ]:
class MergeExperiments(DiceTask):

    n_experiments = RollDiceExperiments.n_experiments
    seed = None

    def requires(self):
        return RollDiceExperiments.req(self)

    def output(self):
        return self.local_target(f"merged_{self.n_sides}sides_{self.n_rolls}rolls_{self.n_experiments}experiments.json")

    def run(self):
        # loop over the outputs of all branches
        means = []
        for inp in self.input()["collection"].targets.values():
            rolls = inp.load()
            means.append(sum(rolls) / len(rolls))

        result = {
            "means": means,
            "mean_of_means": sum(means) / len(means),
            "expected_mean": (self.n_sides + 1) / 2,
        }
        self.output().dump(result, indent=4)

        mean, expected = result["mean_of_means"], result["expected_mean"]
        self.publish_message(f"mean of {len(means)} experiments: {mean:.3f}, expected: {expected}")

In [ ]:
%ilaw run MergeExperiments --print-status 1

In [ ]:
%ilaw run MergeExperiments

`LocalWorkflow` runs its branches in the current process.
The real strength of workflows is that the same task can also submit its branches as jobs to batch systems such as HTCondor or Slurm, by inheriting from the corresponding workflow class, without any change to `output()` and `run()`.

**Try it:** Increase the number of experiments with `--n-experiments 20` and run `MergeExperiments` again.
Only the 10 new branches are processed.

---

## Stage 6: Inspecting and cleaning up

Besides `--print-status` and `--print-output`, `law run` has more **interactive parameters** which inspect or manipulate tasks without running them.

`--print-deps` prints the dependency tree up to a certain depth:

In [ ]:
%ilaw run MergeExperiments --print-deps -1

`--remove-output` removes outputs up to a certain depth.
Its value consists of the depth and a mode:

- `d`: dry run, only shows what would be removed,
- `a`: removes all outputs without asking,
- `i`: asks for each output, which is the default on the command line.

It is good practice to do a dry run first.

In [ ]:
%ilaw run MergeExperiments --remove-output 1,d

In [ ]:
%ilaw run MergeExperiments --remove-output 0,a

Only the output of `MergeExperiments` was removed, since the depth was `0`.
A third value `y` re-runs the task after the removal, which is convenient during development:

In [ ]:
%ilaw run MergeExperiments --remove-output 0,a,y

---

## Stage 7: Writing outputs robustly

Since completeness is defined by the existence of outputs, a task that fails halfway but leaves a partial output behind would be considered complete the next time.

The task below simulates this problem.
It writes its output directly to the final path with plain Python, just like many external tools do, and fails afterwards.

In [ ]:
class FragileTask(TutorialTask):

    fail = luigi.BoolParameter(default=False, description="whether to fail; default: False")

    def output(self):
        return self.local_target("result.txt")

    def run(self):
        # unlike dump(), plain python does not create missing directories, so create the parent directory first
        output = self.output()
        output.parent.touch()

        with open(output.abspath, "w", encoding="utf-8") as f:
            f.write("first half of the result\n")
            if self.fail:
                raise Exception("something went wrong")
            f.write("second half of the result\n")

In [ ]:
%ilaw run FragileTask --fail

The task failed, as you can see in the summary, but its partial output exists.
Running it again, even without `--fail`, does not fix it, because the task already appears complete:

In [ ]:
%ilaw run FragileTask

print(FragileTask().output().load(formatter="text"))

There are two ways to prevent this.

**Decorators**: law provides decorators for `run()` methods.
`@law.decorator.safe_output` removes all outputs when the method raises an exception.

**Localization**: `localize("w")` provides a temporary target, and only moves it to the actual output location when the context is left without an error.
This is the recommended way to pass output paths to external tools.
It also works for remote targets, where the temporary file is uploaded afterwards.

In [ ]:
class RobustTask(TutorialTask):

    fail = luigi.BoolParameter(default=False, description="whether to fail; default: False")

    def output(self):
        return self.local_target("result.txt")

    @law.decorator.safe_output
    def run(self):
        with self.output().localize("w") as tmp, open(tmp.abspath, "w", encoding="utf-8") as f:
            f.write("first half of the result\n")
            if self.fail:
                raise Exception("something went wrong")
            f.write("second half of the result\n")

In [ ]:
%ilaw run RobustTask --fail

print("output exists:", RobustTask().output().exists())

In [ ]:
%ilaw run RobustTask

print(RobustTask().output().load(formatter="text"))

Each of the two mechanisms is sufficient in this case, but they also complement each other, e.g. for tasks with multiple outputs that are written one after another.
Other decorators are, for instance, `@law.decorator.timeit` to measure the run time, `@law.decorator.log` to write the task output into a log file, and `@law.decorator.notify` to send notifications via Slack, Telegram or Mattermost.

---

## Stage 8: Where to go from here

Congratulations, you know the most important concepts of law! 🎉

### From notebooks to the command line

In real projects, tasks are defined in Python modules and run from the command line.
Everything you did with `%ilaw` works the same way with the `law` executable:

1. Put your tasks into a module, e.g. `tasks.py`, inheriting from `law.Task` instead of `law.ipython.Task`.
2. List the module in the `[modules]` section of your `law.cfg` file.
3. Run `law index` once, which makes your tasks and their parameters known to law, including autocompletion in your shell.
4. Run tasks via `law run PlotHistogram --n-rolls 600`.

### Further topics

- **[Sandboxing](https://law.readthedocs.io/en/latest/sandboxing.html)**: run tasks in different software environments, e.g. containers or virtual environments.
- **[Remote workflows](https://law.readthedocs.io/en/latest/remote_workflows.html)**: submit workflow branches as jobs to HTCondor, Slurm, LSF, the WLCG, and more.
- **[Remote targets](https://law.readthedocs.io/en/latest/targets.html)**: store outputs on remote storage, such as EOS, dCache or Dropbox.
- **[Configuration](https://law.readthedocs.io/en/latest/config.html)**: configure law and luigi via the `law.cfg` file.

### Examples

The [examples](https://github.com/riga/law/tree/master/examples) directory of the law repository contains runnable examples for all of these topics, e.g.

- [loremipsum](https://github.com/riga/law/tree/master/examples/loremipsum) and [workflows](https://github.com/riga/law/tree/master/examples/workflows) for projects run from the command line,
- [htcondor_at_cern](https://github.com/riga/law/tree/master/examples/htcondor_at_cern) for remote workflows,
- [docker_sandboxes](https://github.com/riga/law/tree/master/examples/docker_sandboxes) and [subshell_sandboxes](https://github.com/riga/law/tree/master/examples/subshell_sandboxes) for sandboxing, and
- [wlcg_targets](https://github.com/riga/law/tree/master/examples/wlcg_targets) for remote targets.

To start over with this tutorial, simply remove the `data` directory next to this notebook.